# Chapitre 2 — Leçon 4 : Preprocessing pour le Machine Learning

## Objectifs d'apprentissage

À la fin de cette leçon, vous serez capable de :
- **Distinguer** les transformations "safe" (Module 2) des transformations ML (ce chapitre)
- **Utiliser** SimpleImputer pour imputer les valeurs manquantes sans data leakage
- **Appliquer** StandardScaler et MinMaxScaler correctement
- **Encoder** les variables catégorielles avec OneHotEncoder et OrdinalEncoder
- **Discrétiser** avec KBinsDiscretizer (alternative safe à qcut)
- **Créer** un transformer personnalisé pour la winsorisation

---

## 🎯 Accroche : Le piège invisible du Module 2

Dans le Module 2, vous avez appris à nettoyer vos données : supprimer des lignes, traiter les doublons, filtrer les valeurs impossibles. Excellent !

Mais attendez... Vous avez peut-être aussi vu des tutoriels qui proposent :

```python
# ❌ CODE DANGEREUX - Ne faites JAMAIS ça avant le train/test split !
df['age'].fillna(df['age'].mean(), inplace=True)
df['age'] = (df['age'] - df['age'].mean()) / df['age'].std()
df = pd.get_dummies(df, columns=['departement'])
```

**Question :** Pourquoi ces opérations, pourtant courantes, peuvent-elles ruiner votre modèle ML ?

*(Réponse attendue : Ces calculs utilisent des statistiques (moyenne, écart-type) de TOUT le dataset, y compris les données de test → data leakage)*

---

## 4.1 Le pont entre Module 2 et Module 3

### Rappel : Qu'avez-vous fait dans le Module 2 ?

Dans le Module 2, vous avez appris des opérations de nettoyage **"safe"** — c'est-à-dire des opérations qui n'introduisent JAMAIS de data leakage :

```
┌─────────────────────────────────────────────────────────────────────┐
│           MODULE 2 : OPÉRATIONS SAFE (déjà apprises)               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ✅ dropna()           → Supprimer lignes avec NaN                  │
│  ✅ drop(columns=[])   → Supprimer colonnes inutiles               │
│  ✅ drop_duplicates()  → Supprimer doublons                        │
│  ✅ query('age >= 0')  → Filtrer valeurs impossibles (règle fixe)  │
│  ✅ pd.cut(bins=[...]) → Binning avec bornes FIXES                 │
│  ✅ df['date'].dt.year → Extraction temporelle                     │
│  ✅ df['a'] - df['b']  → Calculs ligne par ligne                   │
│                                                                     │
│  → Ces opérations appliquent des RÈGLES FIXES à chaque ligne       │
│  → Aucune statistique n'est calculée sur d'autres lignes           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Ce qui MANQUAIT dans le Module 2

Mais vous n'avez PAS appris à :
- Imputer les NaN avec la moyenne/médiane
- Normaliser les données (scaling)
- Encoder les variables catégorielles (one-hot, ordinal)
- Discrétiser par quantiles (qcut)
- Winsoriser les outliers (capping aux percentiles)

**Pourquoi ?** Parce que ces opérations calculent des **statistiques** sur les données. Et si vous les faites AVANT le train/test split, vous utilisez les données de test pour calculer ces statistiques → **DATA LEAKAGE**.

### Le tableau récapitulatif

| Opération | Module 2 (avant split) | Module 3 (dans Pipeline) |
|-----------|------------------------|---------------------------|
| Supprimer NaN (dropna) | ✅ Safe | - |
| Imputer NaN (fillna/mean) | ❌ Data leakage | ✅ SimpleImputer |
| Supprimer outliers (règle fixe) | ✅ Safe | - |
| Winsoriser outliers (percentiles) | ❌ Data leakage | ✅ Custom Transformer |
| Binning fixe (pd.cut) | ✅ Safe | - |
| Binning quantiles (pd.qcut) | ❌ Data leakage | ✅ KBinsDiscretizer |
| One-hot encoding (get_dummies) | ❌ Data leakage | ✅ OneHotEncoder |
| Normalisation (z-score) | ❌ Data leakage | ✅ StandardScaler |

<details>
<summary>🤔 Question Socratique : Pourquoi pd.get_dummies() est-il considéré comme "unsafe" ?</summary>

### 🔑 Réponse

`pd.get_dummies()` pose deux problèmes pour le ML :

**1. Catégories inconnues en test**
- Si le train a les catégories [A, B, C] et le test a [A, B, D]
- `get_dummies` sur test créera une colonne `D` qui n'existait pas pendant l'entraînement
- Le modèle ne sait pas quoi faire de cette nouvelle colonne

**2. Consistance des colonnes**
- `get_dummies` crée les colonnes basées sur les données présentes
- Train et test peuvent avoir des colonnes différentes
- Impossible de prédire correctement

**Solution : OneHotEncoder**
- `.fit()` apprend les catégories sur le train
- `.transform()` utilise TOUJOURS ces mêmes catégories
- `handle_unknown='ignore'` gère les nouvelles catégories proprement

</details>

In [25]:
# Importations nécessaires
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Les transformers que nous allons apprendre
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.base import BaseEstimator, TransformerMixin

print("✅ Imports chargés")

✅ Imports chargés


In [3]:
# Créer un dataset réaliste pour nos exemples
np.random.seed(42)
# seed for reproducibility for random data
n = 200

df = pd.DataFrame({
    'age': np.concatenate([np.random.randint(20, 65, n-5), [np.nan]*3, [18, 85]]),
    'salaire': np.concatenate([np.random.normal(50000, 15000, n-3), [200000, np.nan, np.nan]]),
    'anciennete': np.random.randint(0, 30, n),
    'departement': np.random.choice(['IT', 'RH', 'Finance', 'Marketing', None], n),
    'niveau_etude': np.random.choice(['Bac', 'Licence', 'Master', 'PhD'], n),
})

# Target
df['promotion'] = ((df['anciennete'] > 5) & (df['salaire'] < 60000)).astype(int)

print("Dataset créé :")
print(df.head(10))
print(f"\nValeurs manquantes :")
print(df.isnull().sum())

Dataset créé :
    age       salaire  anciennete departement niveau_etude  promotion
0  58.0  49721.737932           6     Finance          Bac          1
1  48.0  24897.430670          23          IT          Bac          1
2  34.0  33912.022483          22          IT      Licence          1
3  62.0  35111.207309          29          IT          PhD          1
4  27.0  51535.215238           4        None      Licence          0
5  40.0  43510.860792          11   Marketing          PhD          1
6  58.0  40112.265463          16        None      Licence          1
7  38.0  50059.059566          22   Marketing       Master          1
8  42.0  57166.311492          12        None          PhD          1
9  30.0  46114.570329          22        None       Master          1

Valeurs manquantes :
age              3
salaire          2
anciennete       0
departement     43
niveau_etude     0
promotion        0
dtype: int64


In [4]:
#Séparer features et target AVANT tout preprocessing
X = df.drop('promotion', axis=1)
y = df['promotion']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# X (Matrice) : En mathématiques (algèbre linéaire), on utilise des lettres majuscules pour désigner des Matrices (des tableaux à deux dimensions)
# y (Vecteur) : On utilise des lettres minuscules pour désigner des Vecteurs (des tableaux à une dimension)

print(f"Train : {len(X_train)} exemples")
print(f"Test : {len(X_test)} exemples")
print(f"\n→ À partir de maintenant, on ne touche PLUS à X_test pour calculer des statistiques !")

Train : 160 exemples
Test : 40 exemples

→ À partir de maintenant, on ne touche PLUS à X_test pour calculer des statistiques !


---

## 4.2 SimpleImputer : Imputer sans data leakage

Dans le Module 2, vous avez gardé les NaN pour cette raison. Maintenant, apprenons à les traiter correctement.

### Le pattern fit/transform

```markdown 
┌───────────────────────────────────────────────────────────────────────────┐
│                    SIMPLEIMPUTER : FIT/TRANSFORM                          │
├───────────────────────────────────────────────────────────────────────────┤
│   # Créer le transformateur                                               │
│   imputer = SimpleImputer(strategy='median')                              │    
│                                                                           │
│           ││                                                              │
│                                                                           │
│   # Le fit sur le train                                                   │
│   imputer.fit(X_train)     →  Calcule médiane de X_train : 45000€         │
│                               Stocke cette valeur en mémoire              │
│           ││                                                              │
│                                                                           │
│   # Transformer                                                           │
│   imputer.transform(X_train) →  Remplace NaN par 45000€                   │
│   imputer.transform(X_test)  →  Remplace NaN par 45000€ (MÊME valeur !)   │
│                                                                           │
│           ││                                                              │
│                                                                           │
│   → Le test set utilise les statistiques du train set                     │
│   → PAS de data leakage !                                                 │
│                                                                           │
└───────────────────────────────────────────────────────────────────────────┘
```

Les transformation des colonnes numérque et catégorielles sont différent, donc avant toute transformatoin separer les 2 types de colonnes

In [5]:
# Colonnes numériques avec NaN
col_num = ['age', 'salaire']

# Créer l'imputer
imputer = SimpleImputer(strategy='median')

# FIT sur le train UNIQUEMENT (apprend les médianes)
imputer.fit(X_train[col_num])

print("Médianes apprises sur le train :")
for col, med in zip(col_num, imputer.statistics_):
    print(f"  {col}: {med:.2f}")

Médianes apprises sur le train :
  age: 44.00
  salaire: 53429.31


In [6]:
# TRANSFORM sur train et test (utilise les mêmes médianes)
X_train_imputed = imputer.transform(X_train[col_num])
X_test_imputed = imputer.transform(X_test[col_num])

print("Avant imputation (train) :")
print(f"  NaN dans age: {X_train['age'].isna().sum()}")
print(f"  NaN dans salaire: {X_train['salaire'].isna().sum()}")

print("\nAprès imputation (train) :")
print(f"  NaN dans age: {np.isnan(X_train_imputed[:, 0]).sum()}")
print(f"  NaN dans salaire: {np.isnan(X_train_imputed[:, 1]).sum()}")

Avant imputation (train) :
  NaN dans age: 3
  NaN dans salaire: 2

Après imputation (train) :
  NaN dans age: 0
  NaN dans salaire: 0


### Stratégies disponibles

| Strategy | Description | Quand l'utiliser |
|----------|-------------|------------------|
| `'mean'` | Moyenne | Distribution symétrique |
| `'median'` | Médiane | **Recommandé** - robuste aux outliers |
| `'most_frequent'` | Mode | Variables catégorielles |
| `'constant'` | Valeur fixe | Quand NaN a un sens particulier |

In [7]:
# Imputation catégorielle
imputer_cat = SimpleImputer(strategy='most_frequent')

# On doit transformer en array pour sklearn
X_train_dept = X_train[['departement']].values
X_test_dept = X_test[['departement']].values

imputer_cat.fit(X_train_dept)
print(f"Mode appris : {imputer_cat.statistics_[0]}")

X_train_dept_imputed = imputer_cat.transform(X_train_dept)
print(f"\nNaN avant: {pd.isna(X_train_dept).sum()}")
print(f"NaN après: {pd.isna(X_train_dept_imputed).sum()}")

Mode appris : IT

NaN avant: 35
NaN après: 35


---

## 4.3 Le Scaling

---

### 4.3.1 StandardScaler et MinMaxScaler

#### Pourquoi normaliser ?

Beaucoup d'algorithmes ML sont sensibles à l'échelle des variables :
- **Régression logistique** : les coefficients sont comparés
- **SVM** : calcule des distances
- **KNN** : calcule des distances
- **Réseaux de neurones** : convergence plus rapide

Le ML ne comptend pas la différence de sens entre les âges et les salaires, il va penser que le Salaire est beaucoup plus important que l'âge juste parce que les chiffres sont plus grands.
```
┌─────────────────────────────────────────────────────────────────────┐
│                    POURQUOI NORMALISER ?                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Sans normalisation :                                               │
│    age: [20, 25, 30, 35]        → échelle ~20                       │
│    salaire: [30000, 50000, 80000] → échelle ~50000                  │
│                                                                     │
│  → Le salaire "domine" dans les calculs de distance                 │
│  → L'âge est presque ignoré                                         │
│                                                                     │
│  Avec normalisation (StandardScaler) :                              │ 
│    age: [-1.3, -0.4, 0.4, 1.3]   → échelle ~1                       │
│    salaire: [-1.2, 0.0, 1.2]     → échelle ~1                       │
│                                                                     │
│  → Toutes les variables ont le même poids                           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

#### StandardScaler

In [8]:
# StandardScaler : z-score (moyenne=0, écart-type=1)
scaler = StandardScaler()

# Utilisons les données imputées
scaler.fit(X_train_imputed)  # Apprend moyenne et std du train

print("Statistiques apprises sur le train :")
print(f"  Moyennes: {scaler.mean_}")
print(f"  Écarts-types: {scaler.scale_}")

Statistiques apprises sur le train :
  Moyennes: [4.30875000e+01 5.48039671e+04]
  Écarts-types: [1.36397340e+01 1.81696484e+04]


In [9]:
# Transform
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Avant scaling (train) :")
print(f"  age: min={X_train_imputed[:, 0].min():.1f}, max={X_train_imputed[:, 0].max():.1f}")
print(f"  salaire: min={X_train_imputed[:, 1].min():.0f}, max={X_train_imputed[:, 1].max():.0f}")

print("\nAprès scaling (train) :")
print(f"  age: min={X_train_scaled[:, 0].min():.2f}, max={X_train_scaled[:, 0].max():.2f}")
print(f"  salaire: min={X_train_scaled[:, 1].min():.2f}, max={X_train_scaled[:, 1].max():.2f}")

Avant scaling (train) :
  age: min=18.0, max=85.0
  salaire: min=20076, max=200000

Après scaling (train) :
  age: min=-1.84, max=3.07
  salaire: min=-1.91, max=7.99


Le StandardScaler transforme vos données pour que la moyenne de chaque colonne soit 0 et l'écart-type 1 (réduit la dispersion). 

Cela permet à l'algorithme de ne pas être biaisé par les grandes valeurs et de traiter chaque variable avec la même importance. 

Il écrase les unités d'origine (kilomètres, euros, années, etc.) pour les convertir en une unité commune : la "distance par rapport à la moyenne" (nombre d'écarts-types par rapport à la moyenne).

#### MinMaxScaler : alternative pour valeurs bornées

| Scaler | Formule | Résultat | Quand l'utiliser |
|--------|---------|----------|------------------|
| StandardScaler | (x - μ) / σ | moyenne=0, std=1 | **Par défaut** |
| MinMaxScaler | (x - xmin) / (xmax - xmin) | entre 0 et 1 | Réseaux de neurones, images |

valeur x d'une caractéristique (feature)
μ (mu) est la moyenne de la colonne.
σ (sigma) est l'écart-type (standard deviation) de la colonne.

Le MinMaxScaler est l'outil de choix quand tu veux que tes données tiennent dans un intervalle strictement défini, généralement entre 0 et 1.

In [10]:
# MinMaxScaler
minmax = MinMaxScaler()
minmax.fit(X_train_imputed)

X_train_minmax = minmax.transform(X_train_imputed)

print("Après MinMaxScaler :")
print(f"  age: min={X_train_minmax[:, 0].min():.2f}, max={X_train_minmax[:, 0].max():.2f}")
print(f"  salaire: min={X_train_minmax[:, 1].min():.2f}, max={X_train_minmax[:, 1].max():.2f}")

Après MinMaxScaler :
  age: min=0.00, max=1.00
  salaire: min=0.00, max=1.00


<details>
<summary>🤔 Question Socratique : Pourquoi utiliser les statistiques du train pour transformer le test ?</summary>

#### 🔑 Réponse

Imaginez que vous entraînez un modèle pour prédire les salaires. Votre train set a des salaires entre 30k et 100k, normalisés en [-1, +1].

**Si vous recalculiez les stats sur le test :**
- Test set a salaires entre 40k et 80k
- 80k serait transformé en +1 (le max du test)
- Mais 80k était à ~0.6 dans le train !
- Le modèle reçoit des valeurs incohérentes

**En utilisant les stats du train :**
- 80k est toujours à ~0.6
- Le modèle voit des données cohérentes
- C'est comme ça que le modèle "comprend" les nouvelles données

En production, vous n'aurez qu'une seule donnée à prédire. Vous ne pouvez pas calculer de moyenne sur une seule valeur !

</details>

---

### 4.3.2 Le RobustScaler : L'alternative "Indestructible"

Dans le Module 2, vous avez appris à supprimer les valeurs **impossibles** (âge < 0) avec des règles fixes — c'était safe.

Mais la **winsorisation** (ramener les valeurs extrêmes aux percentiles 5% et 95%) calcule des statistiques → data leakage.

Le StandardScaler est fragile : une seule valeur extrême (un milliardaire parmi des ouvriers) déplace la moyenne et écrase tout le reste des données vers zéro.

Le RobustScaler est l'alternative "blindée" : au lieu d'utiliser la moyenne et l'écart-type, il utilise la Médiane et l'Écart Interquartile (IQR).

**Solution : Un scaling basé sur la majorité, pas sur les extrêmes.**

```
┌─────────────────────────────────────────────────────────────────────┐
│                    ROBUSTSCALER : FIT/TRANSFORM                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   scaler = RobustScaler()                                           │
│                                                                     │
│   .fit(X_train)     →  Calcule la Médiane et l'IQR sur X_train      │
│                        Ex: Médiane=30 ans, IQR=15 ans               │
│                                                                     │
│   .transform(X)     →  Centre sur 0 (Médiane) et divise par l'IQR   │
│                        30 ans  →  0.0  (C'est la médiane)           │
│                        45 ans  →  1.0  (C'est 1 "distance" IQR)     │
│                        150 ans →  8.0  (L'outlier reste loin, mais  │
│                                         n'a pas faussé le calcul)   │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [26]:
# Initialisation
scaler_robust = RobustScaler()

# Fit sur le train (on apprend la médiane et l'IQR du train)
scaler_robust.fit(X_train_imputed)

print("Statistiques robustes apprises :")
print(f"  Médianes (centres) : {scaler_robust.center_}")
print(f"  Écarts IQR (échelles) : {scaler_robust.scale_}")

Statistiques robustes apprises :
  Médianes (centres) : [4.40000000e+01 5.34293149e+04]
  Écarts IQR (échelles) : [   22.5        19264.72283511]


In [27]:
# Transform
X_train_scaled = scaler_robust.transform(X_train_imputed)
X_test_scaled = scaler_robust.transform(X_test_imputed)

print("Avant scaling (train) :")
print(f"  age: min={X_train_imputed[:, 0].min():.1f}, max={X_train_imputed[:, 0].max():.1f}")
print(f"  salaire: min={X_train_imputed[:, 1].min():.0f}, max={X_train_imputed[:, 1].max():.0f}")

print("\nAprès scaling (train) :")
print(f"  age: min={X_train_scaled[:, 0].min():.2f}, max={X_train_scaled[:, 0].max():.2f}")
print(f"  salaire: min={X_train_scaled[:, 1].min():.2f}, max={X_train_scaled[:, 1].max():.2f}")

Avant scaling (train) :
  age: min=18.0, max=85.0
  salaire: min=20076, max=200000

Après scaling (train) :
  age: min=-1.16, max=1.82
  salaire: min=-1.73, max=7.61


### Le Guide du Scaling

#### 1. La règle de base : Différentes échelles = Scaler obligatoire

Dès que tes colonnes n'ont pas la même unité (ex: Age vs Salaire), tu dois passer par un scaler pour les modèles sensibles à la distance (Régression, KNN, SVM, Réseaux de neurones).

#### 2. Le choix du Scaler selon les Outliers

Voici le raisonnement à tenir :

* **Scénario A : Pas d'outliers + Distribution "normale" (en cloche)**
    *   Outil : `StandardScaler`
    *   Pourquoi : C'est le plus performant mathématiquement si tes données sont propres.


* **Scénario B : Pas d'outliers + Données bornées (ex: Pixels 0-255)**
    * Outil : `MinMaxScaler`
    * Pourquoi : On veut garder les données dans un intervalle fixe [0, 1].


* **Scénario C : Présence d'outliers (le "champ de mines")**
    * Outil : `RobustScaler`
    * Pourquoi : Le `StandardScaler` serait "décalé" par les outliers. Le `RobustScaler`, lui, les ignore pour calculer son échelle.



#### Pourquoi le RobustScaler est "supérieur" en cas d'outliers ?

Imagine que tu calcules la taille moyenne d'une classe de 20 étudiants :

1. Si tout le monde fait environ 1m75, la **moyenne** est fiable.
2. Si un géant de 5 mètres rentre dans la salle, la **moyenne** grimpe à 2m10. Le `StandardScaler` va alors croire que tes étudiants de 1m75 sont "petits" par rapport à cette moyenne faussée.

**Le RobustScaler, lui, utilise la Médiane :**
Le géant de 5 mètres ne change pas la **médiane**. Elle reste à 1m75. Le scaler reste donc parfaitement calibré pour la majorité des élèves.



#### Tableau de décision

| Présence d'outliers ? | Échelles différentes ? | Scaler recommandé |
| --- | --- | --- |
| ❌ Non | ✅ Oui | **StandardScaler** |
| ❌ Non (et borné) | ✅ Oui | **MinMaxScaler** |
| ✅ **Oui** | ✅ **Oui** | **RobustScaler** |



> **Attention :** certains modèles comme les **Arbres de décision (Random Forest, XGBoost)** se moquent complètement des échelles. Ils n'ont techniquement pas besoin de scaler. Mais dans le doute, en mettre un (surtout un Robust) ne fera jamais de mal à tes performances !

---

## 4.4 Encodage des variables catégorielles

Les modèles ML ne comprennent que les **valeurs numériques**. Ils ne peuvent pas interpréter directement 'Nord', 'Sud', etc. Il faut donc transformer les catégories textuelles.

```
┌─────────────────────────────────────────────────────────────────────┐
│              TYPES D'ENCODAGE CATÉGORIEL                           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ORDINAL ENCODER (quand ordre existe)                              │
│  ─────────────────────────────────────                             │
│  'Bac' → 0, 'Licence' → 1, 'Master' → 2, 'PhD' → 3                │
│  → Préserve l'ordre naturel                                        │
│  → Pour : niveau d'étude, taille (S/M/L), rating (1-5)            │
│                                                                     │
│  ONE-HOT ENCODER (quand pas d'ordre)                               │
│  ───────────────────────────────────                               │
│  'IT' → [1,0,0,0], 'RH' → [0,1,0,0], 'Finance' → [0,0,1,0]        │
│  → Pas de relation d'ordre artificielle                            │
│  → Pour : département, pays, couleur                               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### OneHotEncoder

In [11]:
# OrdinalEncoder pour niveau_etude (ordre naturel)
ordinal_encoder = OrdinalEncoder(
    categories=[['Bac', 'Licence', 'Master', 'PhD']]  # Ordre explicite !
)
# categories : On définit l'ordre des catégories pour l'encodage ordinal


ordinal_encoder.fit(X_train[['niveau_etude']])
# l'encodeur regarde la colonne dans le set d'entraînement. 
# Il vérifie que les données présentes correspondent bien aux catégories définies 
# et il prépare la table de conversion interne.

X_train_niveau = ordinal_encoder.transform(X_train[['niveau_etude']])
X_test_niveau = ordinal_encoder.transform(X_test[['niveau_etude']])

print("Encoding ordinal :")
print(f"  Catégories: {ordinal_encoder.categories_[0]}")
# categories_ : Attribut qui contient les catégories apprises par l'encodeur
# _ C'est un attribut de Scikit-Learn qui n'existe qu'après le fit

print(f"  Bac → {ordinal_encoder.transform([['Bac']])[0][0]}")
print(f"  PhD → {ordinal_encoder.transform([['PhD']])[0][0]}")

Encoding ordinal :
  Catégories: ['Bac' 'Licence' 'Master' 'PhD']
  Bac → 0.0
  PhD → 3.0


/Users/safae/.universal_kernel/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
/Users/safae/.universal_kernel/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(


### OrdinalEncoder

In [12]:
# OneHotEncoder pour departement (pas d'ordre)
onehot_encoder = OneHotEncoder(
    sparse_output=False,      # Retourne array dense (pas sparse matrix)
    handle_unknown='ignore'   # Ignore les catégories inconnues en test
)

# D'abord imputer les NaN
X_train_dept_clean = imputer_cat.transform(X_train[['departement']].values)
X_test_dept_clean = imputer_cat.transform(X_test[['departement']].values)

onehot_encoder.fit(X_train_dept_clean)

X_train_dept_encoded = onehot_encoder.transform(X_train_dept_clean)
X_test_dept_encoded = onehot_encoder.transform(X_test_dept_clean)

print("Encoding one-hot :")
print(f"  Catégories apprises: {onehot_encoder.categories_[0]}")
print(f"  Nombre de colonnes créées: {X_train_dept_encoded.shape[1]}")
print(f"\nExemple (première ligne) :")
print(f"  {X_train['departement'].iloc[0]} → {X_train_dept_encoded[0]}")

Encoding one-hot :
  Catégories apprises: ['Finance' 'IT' 'Marketing' 'RH' None]
  Nombre de colonnes créées: 5

Exemple (première ligne) :
  RH → [0. 0. 0. 1. 0.]


#### ***Exemple avec get dummies pour visualisation***

In [24]:
import pandas as pd

# 1. Données brutes
df = pd.DataFrame({
    'client_id': [1, 2, 3, 4, 5],
    'region': ['Nord', 'Sud', 'Est', 'Ouest', 'Nord'],
    'segment': ['Premium', 'Standard', 'Premium', 'Basic', 'Standard']
})

#Methode classique avec pd.get_dummies
df_encoded = pd.get_dummies(df, columns=['region'], prefix='reg')
print("One-Hot Encoding avec pd.get_dummies :")
print(df_encoded)
print(" ")
print("="*10)

# Méthode détaillée étape par étape

# 2. On génère le One-Hot (en gardant les colonnes séparées)
df_ohe = pd.get_dummies(df['region'], prefix='reg').astype(int)
print("One-Hot Encoding :")
print(df_ohe)

# 3. On crée la colonne "Vecteur" pour la visualisation
# On regroupe les colonnes reg_ en une liste pour chaque ligne
df_ohe['Vecteur_Final'] = df_ohe.values.tolist()

# 4. Fusion pour affichage final
df_final = pd.concat([df, df_ohe], axis=1)

print("="*10)
print("Final DataFrame with One-Hot Encoding:")
print(df_final)

One-Hot Encoding avec pd.get_dummies :
   client_id   segment  reg_Est  reg_Nord  reg_Ouest  reg_Sud
0          1   Premium    False      True      False    False
1          2  Standard    False     False      False     True
2          3   Premium     True     False      False    False
3          4     Basic    False     False       True    False
4          5  Standard    False      True      False    False
 
One-Hot Encoding :
   reg_Est  reg_Nord  reg_Ouest  reg_Sud
0        0         1          0        0
1        0         0          0        1
2        1         0          0        0
3        0         0          1        0
4        0         1          0        0
Final DataFrame with One-Hot Encoding:
   client_id region   segment  reg_Est  reg_Nord  reg_Ouest  reg_Sud  \
0          1   Nord   Premium        0         1          0        0   
1          2    Sud  Standard        0         0          0        1   
2          3    Est   Premium        1         0          0        

**Question :** Pourquoi `handle_unknown='ignore'` est-il important ?

*(Réponse attendue : Si une nouvelle catégorie apparaît en test/production, au lieu de planter, le encoder met simplement tous les zéros pour cette ligne)*

### Pourquoi des doubles crochets [[...]] ?

Dans Scikit-Learn, la règle est la suivante :

* **Les "Features" (X)** doivent toujours être un tableau en **2D** (une matrice avec des lignes et des colonnes). Même si il n'y a qu'une seule colonne comme `niveau_etude`, l'algorithme attend une structure "tableau".
* `X_train['niveau_etude']` (un seul crochet) renvoie une **Série** (1D).
* `X_train[['niveau_etude']]` (doubles crochets) renvoie un **DataFrame** (2D).

L'argument `categories` attend une **liste de listes**.

* **La liste extérieure :** Elle contient une entrée pour chaque colonne que tu veux encoder.
* **La liste intérieure :** Elle contient l'ordre des catégories pour *cette* colonne spécifique.

Si on encode deux colonnes d'un coup (par exemple `niveau_etude` et `taille_vetement`), la syntaxe ressemblerait à ceci :

```python
categories=[
    ['Bac', 'Licence', 'Master', 'PhD'], # Ordre pour la colonne 1
    ['S', 'M', 'L', 'XL']                # Ordre pour la colonne 2
]

```

***Que se passe-t-il si il n'y a pas l'ordre ?***

Avec simplement `ordinal_encoder = OrdinalEncoder()`, c'est la **convention** qui prend le relais :

* Scikit-Learn va utiliser l'ordre **alphabétique**.
* Problème : "Bac" (B) viendrait avant "Licence" (L), mais "Master" (M) viendrait après "Licence", et "PhD" (P) serait à la fin.
* Ici, l'ordre alphabétique respecte presque la logique, mais pour des tailles comme `['Grand', 'Medium', 'Petit']`, l'ordre alphabétique donnerait `G, M, P`, ce qui est faux logiquement.

---

## 4.5 KBinsDiscretizer : Alternative safe à qcut

Dans le Module 2, vous avez appris `pd.cut()` avec des **bornes fixes** — c'était safe.

Mais `pd.qcut()` (qui crée des bins par **quantiles**) calcule les quantiles sur les données → data leakage si fait avant le split.

**Solution : KBinsDiscretizer**

c'est l'outil que l'on utilise quand on veut transformer une variable numérique continue (comme l'âge ou le revenu) en une variable catégorielle ordinale.

### Explication

**Certains modèles sont "rigides".** Ils cherchent une règle simple (souvent une ligne droite) pour séparer les données.

**1. L'exemple de "L'effet de l'âge"**

Imagine que tu veuilles prédire si quelqu'un va acheter un jouet.

* Les **enfants** (0-12 ans) en achètent beaucoup.
* Les **adultes** (25-40 ans) en achètent pour leurs enfants.
* Les **étudiants** (18-24 ans) n'en achètent pas du tout.

Si tu donnes l'âge brut (colonne numérique de 0 à 80) à une **Régression Logistique** :
Le modèle va essayer de trouver une logique du type : *"Plus on vieillit, plus on achète"* (ligne montante) ou *"Plus on vieillit, moins on achète"* (ligne descendante).

**Le problème :** Ici, la relation fait des "vagues" (ça monte, ça descend, ça remonte). La Régression Logistique est incapable de voir ces vagues, elle ne voit que des lignes droites. C'est ça, une **relation non-linéaire**.

**2. Comment le KBinsDiscretizer "aide" le modèle ?**

Si tu découpes l'âge en tranches avec le `KBinsDiscretizer` :

* Tranche 0 : [0-12 ans]
* Tranche 1 : [13-25 ans]
* Tranche 2 : [26-45 ans]

Et que tu appliques un **One-Hot Encoding** sur ces tranches, le modèle reçoit maintenant 3 colonnes distinctes. Il peut alors donner un "poids" différent à chaque boîte :

* Un poids **fort** pour la boîte [0-12].
* Un poids **faible** pour la boîte [13-25].
* Un poids **fort** pour la boîte [26-45].

**Résultat :** En découpant la donnée, tu as transformé une "vague" complexe en une série de "marches d'escalier" que le modèle peut enfin comprendre.

### Code

In [15]:
# KBinsDiscretizer avec stratégie 'quantile'
discretizer = KBinsDiscretizer(
    n_bins=4,              # Nombre de bins
    encode='ordinal',       # 0, 1, 2, 3 (pas one-hot)
    strategy='quantile'     # Comme qcut !
)

# Fit sur train uniquement !
discretizer.fit(X_train_imputed[:, [1]])  # Colonne salaire

print("Bornes de bins apprises (salaire) :")
print(f"  {discretizer.bin_edges_[0]}")

Bornes de bins apprises (salaire) :
  [ 20075.57642664  44431.69141447  53429.31494333  63696.41424958
 200000.        ]


In [16]:
# Transform
salaire_train_binned = discretizer.transform(X_train_imputed[:, [1]])
salaire_test_binned = discretizer.transform(X_test_imputed[:, [1]])

print("Distribution des bins (train) :")
unique, counts = np.unique(salaire_train_binned, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Bin {int(u)}: {c} exemples ({c/len(salaire_train_binned)*100:.1f}%)")

Distribution des bins (train) :
  Bin 0: 40 exemples (25.0%)
  Bin 1: 39 exemples (24.4%)
  Bin 2: 41 exemples (25.6%)
  Bin 3: 40 exemples (25.0%)


### Stratégies de binning

| Strategy | Description | Équivalent pandas |
|----------|-------------|-------------------|
| `'uniform'` | Bins de même largeur (ex: 0-25, 25-50, 50-75, 75-100) | `pd.cut(bins=n)` |
| `'quantile'` | Bins avec même nombre d'exemples Si tu as 100 clients et que tu veux 4 tranches, il y aura 25 clients dans chaque panier, peu importe l'écart d'âge| `pd.qcut()` |
| `'kmeans'` | Bins basés sur clustering pour regrouper les valeurs qui se ressemblent naturellement| - |

### Question : Quelle est la différence avec le simple Scaling ? 

Le Scaling change l'échelle, mais le Discretizer change la nature même de la donnée (de continue à catégorielle).

---

## 4.6 Le raccourci : `.fit_transform()`

Pour le train set, on fait souvent `.fit()` puis `.transform()`. Le raccourci `.fit_transform()` combine les deux :

In [28]:
# Méthode longue (équivalente)
scaler1 = StandardScaler()
scaler1.fit(X_train_imputed)
X_train_v1 = scaler1.transform(X_train_imputed)

# Méthode courte (raccourci)
scaler2 = StandardScaler()
X_train_v2 = scaler2.fit_transform(X_train_imputed)  # fit + transform en une ligne

# Vérifions que c'est identique
print(f"Résultats identiques ? {np.allclose(X_train_v1, X_train_v2)}")

Résultats identiques ? True


⚠️ **Attention !** N'utilisez **jamais** `.fit_transform()` sur le test set !

```python
# ✅ CORRECT
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)         # SEULEMENT transform

# ❌ ERREUR (data leakage !)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)     # fit sur test = LEAKAGE
```

---

## 4.7 Tableau récapitulatif : De Module 2 à sklearn

| Ce que vous faisiez (Module 2) | Ce que vous devez faire (Module 3) |
|-------------------------------|------------------------------------|
| `df['col'].fillna(df['col'].mean())` | `SimpleImputer(strategy='mean')` |
| `df['col'].fillna(df['col'].median())` | `SimpleImputer(strategy='median')` |
| `(df['col'] - df['col'].mean()) / df['col'].std()` | `StandardScaler()` |
| `(df['col'] - df['col'].min()) / (df['col'].max() - df['col'].min())` | `MinMaxScaler()` |
| `pd.get_dummies(df, columns=['col'])` | `OneHotEncoder(handle_unknown='ignore')` |
| Mapping manuel {A:0, B:1, C:2} | `OrdinalEncoder(categories=[[...]])` |
| `pd.qcut(df['col'], q=4)` | `KBinsDiscretizer(n_bins=4, strategy='quantile')` |
| Winsorisation | `RobustScaler() |

**Règle d'or :** Si l'opération calcule une statistique (moyenne, médiane, percentile, etc.), utilisez un transformer sklearn et faites `.fit()` sur le train uniquement.

---

## ✍️ Exercice 4.1 : Preprocessing complet

Appliquez le preprocessing complet sur ce dataset :

In [20]:
# Dataset d'exercice
np.random.seed(123)
df_ex = pd.DataFrame({
    'revenus': np.concatenate([np.random.normal(45000, 12000, 95), [np.nan]*3, [150000, 200000]]),
    'age': np.concatenate([np.random.randint(22, 60, 98), [np.nan, np.nan]]),
    'experience': np.random.choice(['Junior', 'Confirmé', 'Senior', 'Expert'], 100),
    'secteur': np.random.choice(['Tech', 'Finance', 'Santé', None], 100),
})

# Target
df_ex['promotion'] = np.random.randint(0, 2, 100)

print("Dataset :")
print(df_ex.head())
print(f"\nValeurs manquantes : {df_ex.isnull().sum().sum()}")

Dataset :
        revenus   age experience  secteur  promotion
0  31972.432760  23.0     Senior    Santé          1
1  56968.145359  36.0     Senior  Finance          0
2  48395.741977  45.0     Expert  Finance          0
3  26924.463433  55.0     Senior     Tech          0
4  38056.796976  25.0     Expert     None          1

Valeurs manquantes : 34


In [ ]:
# TODO: Complétez le preprocessing

# 1. Séparation features/target et train/test split
X_ex = df_ex.drop('promotion', axis=1)
y_ex = df_ex['promotion']

X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(
    X_ex, y_ex, test_size=0.2, random_state=42
)

# 2. Définir les colonnes
col_num = ['revenus', 'age']
col_ord = ['experience']  # Ordinal : Junior < Confirmé < Senior < Expert
col_cat = ['secteur']     # Catégoriel sans ordre

# 3. Preprocessing numérique
# - Imputer avec médiane
# - Scaler pour les outliers
imputer_num = SimpleImputer(strategy='median')
scaler_num = RobustScaler()

# Fit et transform
X_train_num = imputer_num.fit_transform(X_train_ex[col_num])
X_train_num = scaler_num.fit_transform(X_train_num)

X_test_num = imputer_num.transform(X_test_ex[col_num])
X_test_num = scaler_num.transform(X_test_num)

print("✅ Preprocessing numérique terminé")
print(f"   Shape train: {X_train_num.shape}")

✅ Preprocessing numérique terminé
   Shape train: (80, 2)


In [22]:
# 4. Preprocessing ordinal
ordinal_enc = OrdinalEncoder(
    categories=[['Junior', 'Confirmé', 'Senior', 'Expert']]
)

X_train_ord = ordinal_enc.fit_transform(X_train_ex[col_ord])
X_test_ord = ordinal_enc.transform(X_test_ex[col_ord])

print("✅ Preprocessing ordinal terminé")

# 5. Preprocessing catégoriel
imputer_cat = SimpleImputer(strategy='most_frequent')
onehot_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_cat = imputer_cat.fit_transform(X_train_ex[col_cat])
X_train_cat = onehot_enc.fit_transform(X_train_cat)

X_test_cat = imputer_cat.transform(X_test_ex[col_cat])
X_test_cat = onehot_enc.transform(X_test_cat)

print("✅ Preprocessing catégoriel terminé")
print(f"   Colonnes créées: {onehot_enc.get_feature_names_out()}")

✅ Preprocessing ordinal terminé
✅ Preprocessing catégoriel terminé
   Colonnes créées: ['x0_Finance' 'x0_Santé' 'x0_Tech' 'x0_None']


In [ ]:
# 6. Combiner toutes les features
X_train_final = np.hstack([X_train_num, X_train_ord, X_train_cat])
X_test_final = np.hstack([X_test_num, X_test_ord, X_test_cat])

# hstack : empile horizontalement (colonnes)

print("✅ Features combinées")
print(f"   Shape finale train: {X_train_final.shape}")
print(f"   Shape finale test: {X_test_final.shape}")

✅ Features combinées
   Shape finale train: (80, 7)
   Shape finale test: (20, 7)


---

## 🧠 Réflexion métacognitive

1. **Pouvez-vous expliquer** à un collègue pourquoi `df.fillna(df.mean())` est dangereux pour le ML ?

2. **Quelle est la différence** entre StandardScaler et MinMaxScaler ? Quand utiliser l'un ou l'autre ?

3. **Dans votre prochain projet**, comment allez-vous organiser votre preprocessing ?

---

## 📝 Résumé

| Transformer | Utilisation | Pattern |
|-------------|-------------|----------|
| `SimpleImputer` | Remplacer les NaN | `.fit(X_train)` apprend stats, `.transform()` applique |
| `StandardScaler` | Normaliser (μ=0, σ=1) | Idem |
| `MinMaxScaler` | Normaliser [0, 1] | Idem |
| `OneHotEncoder` | Catégories → binaire | Idem |
| `OrdinalEncoder` | Catégories → entiers ordonnés | Idem |
| `KBinsDiscretizer` | Continu → bins | Idem |
| `RobustScaler` | Winsorisation | Normalisation pour outlier |

**Pattern universel :**
```python
transformer.fit(X_train)       # Apprend sur train
X_train_t = transformer.transform(X_train)
X_test_t = transformer.transform(X_test)  # Utilise stats du train !
```

---

## ➡️ Prochaine leçon

Maintenant que vous maîtrisez chaque transformer individuellement, la **Leçon 2.5 : Pipelines scikit-learn** vous montrera comment les **combiner** en un seul objet pour :
- Éviter les erreurs manuelles
- Automatiser le fit/transform
- Sauvegarder tout le workflow en un fichier

**Question de transition :** Combien de lignes de code fallait-il pour faire le preprocessing de l'exercice ? Et si vous pouviez faire tout ça en 3 lignes ?